# 03 · Filter & Rank — the shared multi-layer filter, on BOTH states

**Standard slot:** *filter & rank* via `shared/filtering_pipeline.py` — the same module all 25
projects use. **For Project 22** a switch must fold to **both** states, so you build **two**
`fp.Design` objects per design (one for state A, one for state B), each with
`design_type="monomer"`, and a design only counts as a switch candidate if it passes the monomer
foldability bar for **A AND B** (D3 part 1).

Run `00`–`02` first so `results/multistate_designs.csv` exists.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Load the shared filtering pipeline
This is the cohort's shared module — improvements here are pull-requested back for everyone. We apply
its **monomer** cutoffs to each state independently.

In [ ]:
import filtering_pipeline as fp
import pandas as pd

print("DEFAULT_CUTOFFS:")
for k, v in fp.DEFAULT_CUTOFFS.items():
    print(" ", k, v)
print("\nApplying design_type='monomer' to EACH of the two states.")

## Build `fp.Design` objects for BOTH states

For each row of `multistate_designs.csv` we create two `fp.Design` records — `..._A` and `..._B` —
populating each state's per-state `scrmsd` and `plddt`. We then run the pipeline **once per state**
and intersect: a switch candidate must pass **A AND B**.

In [ ]:
df = pd.read_csv("results/multistate_designs.csv")

designs_a, designs_b = [], []
for _, r in df.iterrows():
    designs_a.append(fp.Design(
        design_id=f"{r['design_id']}__A", sequence=str(r["sequence"]),
        design_type="monomer", scrmsd=r.get("scrmsd_a"), plddt=r.get("plddt_a"),
        extra={"state": "A", "energy_gap": r.get("energy_gap"), "switchable": r.get("switchable")}))
    designs_b.append(fp.Design(
        design_id=f"{r['design_id']}__B", sequence=str(r["sequence"]),
        design_type="monomer", scrmsd=r.get("scrmsd_b"), plddt=r.get("plddt_b"),
        extra={"state": "B", "energy_gap": r.get("energy_gap"), "switchable": r.get("switchable")}))
print(len(designs_a), "state-A Design objects |", len(designs_b), "state-B Design objects")

## Run the pipeline on each state

We use layer 1 (self-consistency: scRMSD + pLDDT) since the campaign CSV carries per-state scRMSD and
pLDDT (no orthogonal/physics columns yet — add those layers once you compute them). `fp.report`
prints the hit-rate accounting and the survival-at-each-layer figure for **each** state.

In [ ]:
ranked_a = fp.run_pipeline(designs_a, design_type="monomer", use_layers=(1,))
ranked_b = fp.run_pipeline(designs_b, design_type="monomer", use_layers=(1,))

print("===== STATE A =====")
top_a = fp.report(ranked_a, top_n=10, save_prefix="results/proj22_state_A")
print("\n===== STATE B =====")
top_b = fp.report(ranked_b, top_n=10, save_prefix="results/proj22_state_B")
ranked_a.to_csv("results/ranked_state_A.csv", index=False)
ranked_b.to_csv("results/ranked_state_B.csv", index=False)
print("\nwrote results/ranked_state_A.csv and results/ranked_state_B.csv")

## Intersect: a switch candidate passes A **AND** B

This is the move that makes it a *switch* filter rather than two monomer filters. Pass requires
`layers_passed >= 1` for **both** states (and, for a real switch, the energy gap inside the band).

In [ ]:
pa = ranked_a.set_index("design_id")["layers_passed"]
pb = ranked_b.set_index("design_id")["layers_passed"]

both = []
for _, r in df.iterrows():
    ok_a = int(pa.get(f"{r['design_id']}__A", 0)) >= 1
    ok_b = int(pb.get(f"{r['design_id']}__B", 0)) >= 1
    both.append(dict(design_id=r["design_id"], pass_a=ok_a, pass_b=ok_b,
                     pass_both=ok_a and ok_b, energy_gap=r.get("energy_gap"),
                     switchable=bool(r.get("switchable")) if pd.notna(r.get("switchable")) else False))
switch_df = pd.DataFrame(both)
switch_df.to_csv("results/switch_candidates.csv", index=False)

n = len(switch_df)
print(f"N total            : {n}")
print(f"pass state A       : {int(switch_df['pass_a'].sum())}")
print(f"pass state B       : {int(switch_df['pass_b'].sum())}")
print(f"pass BOTH (switch) : {int(switch_df['pass_both'].sum())}")
print(f"  ...and switchable: {int((switch_df['pass_both'] & switch_df['switchable']).sum())}")
print("\nThe honest hit-rate ladder: N(A) / N(B) / N(BOTH) / N(switchable).")
print("[mock numbers are SYNTHETIC EXAMPLE_DATA]")
switch_df.head(10)

## D3 (part 1) checklist
- [ ] `fp.Design` built for **both** states; `design_type="monomer"` for each.
- [ ] `fp.run_pipeline(..., design_type="monomer")` + `fp.report(...)` run **per state** (survival-at-each-layer for A and B).
- [ ] Switch candidates = pass A **AND** B, saved to `results/switch_candidates.csv`.
- [ ] Honest hit-rate ladder reported: N(A) / N(B) / N(BOTH) / N(switchable).

**Next:** `04_validate.ipynb` — AF2 predicts both states from one sequence + the energy-gap study + single- vs multi-state benchmark.